In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

In [ ]:
# Utility functions for BB84

backend = BasicSimulator()

def measure_bit(circuit):
    compiled = transpile(circuit, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    counts = result.get_counts(compiled)
    return int(list(counts.keys())[0])

def random_bits(n):
    bits = []
    for i in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        bits = bits + [measure_bit(qc)]
    return bits

def prepare_bb84_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

def measure_bb84_qubit(bit, alice_basis, bob_basis):
    qc = prepare_bb84_qubit(bit, alice_basis)
    if bob_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    return measure_bit(qc)

In [ ]:
# BB84 without an attacker

n = 20

# Alice chooses random bits and random bases.
alice_bits = random_bits(n)
alice_bases = random_bits(n)

# Bob chooses random bases.
bob_bases = random_bits(n)

# Bob measures each qubit that Alice sends.
bob_results = []
for i in range(n):
    bob_results = bob_results + [measure_bb84_qubit(alice_bits[i], alice_bases[i], bob_bases[i])]

# Alice and Bob compare bases publicly and keep only matching positions.
matching_positions = []
for i in range(n):
    if alice_bases[i] == bob_bases[i]:
        matching_positions = matching_positions + [i]

alice_key = [alice_bits[i] for i in matching_positions]
bob_key = [bob_results[i] for i in matching_positions]

print("0 = computational basis, 1 = diagonal basis")
print("Alice bits:  ", alice_bits)
print("Alice bases: ", alice_bases)
print("Bob bases:   ", bob_bases)
print("Bob results: ", bob_results)
print("Matching positions: ", matching_positions)
print("Alice key: ", alice_key)
print("Bob key:   ", bob_key)
print("Keys match: ", alice_key == bob_key)

plot_histogram({"kept bits": len(matching_positions), "discarded bits": n - len(matching_positions)})